# Privileged Brain — Google Colab Fine-Tuning

Fine-tunes **Qwen 2.5 Coder 1.5B** on NL→Bash data using LoRA + SFT, then optionally DPO.
Checkpoints save to Google Drive every 200 steps — safe against disconnects.

| GPU | Est. SFT time (3 epochs, 28k examples) |
|-----|----------------------------------------|
| T4 (free tier) | ~1.5 hrs |
| V100 (Pro) | ~50 min |
| A100 40GB (Pro) | ~25 min |
| A100 80GB (Pro+) | ~15 min |

**Before running:** upload `train.jsonl` and `valid.jsonl` from `privileged-brain/data/processed/` to Google Drive.
See `COLAB_INSTRUCTIONS.md` for the full walkthrough.

In [ ]:
# Cell 1 — Install dependencies (run once per session, ~2 min)
!pip install -q transformers trl peft datasets accelerate huggingface_hub
print("Installation complete.")

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# Cell 3 — Auto-detect GPU and set training flags
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Go to Runtime → Change runtime type → GPU.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {gpu_mem_gb:.1f} GB")

if "A100" in gpu_name and gpu_mem_gb > 60:
    USE_FP16, USE_BF16 = False, True
    BATCH_SIZE, GRAD_ACCUM = 16, 1
    print("→ A100 80GB: bf16, batch=16")
elif "A100" in gpu_name:
    USE_FP16, USE_BF16 = False, True
    BATCH_SIZE, GRAD_ACCUM = 8, 2
    print("→ A100 40GB: bf16, batch=8, grad_accum=2")
elif "V100" in gpu_name:
    USE_FP16, USE_BF16 = True, False
    BATCH_SIZE, GRAD_ACCUM = 4, 4
    print("→ V100: fp16, batch=4, grad_accum=4")
elif "T4" in gpu_name:
    USE_FP16, USE_BF16 = True, False
    BATCH_SIZE, GRAD_ACCUM = 4, 4
    print("→ T4: fp16, batch=4, grad_accum=4")
else:
    USE_FP16, USE_BF16 = True, False
    BATCH_SIZE, GRAD_ACCUM = 2, 8
    print(f"→ Unknown GPU ({gpu_name}): conservative settings")

print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# Cell 4 — Configure paths  ← EDIT THESE if you put files elsewhere on Drive
import os

# Folder containing train.jsonl and valid.jsonl
DRIVE_DATA_DIR   = "/content/drive/MyDrive/privileged-brain/data/processed"

# Folder where adapters and checkpoints will be saved
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/privileged-brain/training"

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
EPOCHS   = 3
LORA_RANK = 8
MAX_SEQ_LEN = 512

os.makedirs(f"{DRIVE_OUTPUT_DIR}/adapters/sft", exist_ok=True)
os.makedirs(f"{DRIVE_OUTPUT_DIR}/adapters/dpo", exist_ok=True)
print(f"Data   : {DRIVE_DATA_DIR}")
print(f"Output : {DRIVE_OUTPUT_DIR}")

In [ ]:
# Cell 5 — Verify data files exist
from pathlib import Path

train_path = Path(DRIVE_DATA_DIR) / "train.jsonl"
valid_path = Path(DRIVE_DATA_DIR) / "valid.jsonl"

assert train_path.exists(), (
    f"\nERROR: {train_path} not found.\n"
    "Upload privileged-brain/data/processed/train.jsonl to Drive first."
)
assert valid_path.exists(), (
    f"\nERROR: {valid_path} not found.\n"
    "Upload privileged-brain/data/processed/valid.jsonl to Drive first."
)

train_count = sum(1 for _ in open(train_path))
valid_count = sum(1 for _ in open(valid_path))
print(f"train.jsonl : {train_count:,} examples  OK")
print(f"valid.jsonl : {valid_count:,} examples  OK")

In [ ]:
# Cell 6 — Load tokenizer and model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.enable_input_require_grads()
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# Cell 7 — Apply LoRA
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=[
        "q_proj", "v_proj", "k_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 8 — Load and format datasets
from datasets import load_dataset

train_ds = load_dataset("json", data_files=str(train_path), split="train")
valid_ds = load_dataset("json", data_files=str(valid_path), split="train")

def format_example(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_ds = train_ds.map(format_example, remove_columns=["messages"])
valid_ds = valid_ds.map(format_example, remove_columns=["messages"])
print(f"Train: {len(train_ds):,} | Valid: {len(valid_ds):,}")

In [ ]:
# Cell 9 — SFT Training
# Checkpoints go to Drive every 200 steps.
# If disconnected: re-run cells 1-8 (skip 6), then re-run this cell — it resumes automatically.
from trl import SFTConfig, SFTTrainer

SFT_DIR = f"{DRIVE_OUTPUT_DIR}/adapters/sft"

# Check if a checkpoint exists to resume from
import glob
checkpoints = sorted(glob.glob(f"{SFT_DIR}/checkpoint-*"))
resume_from = checkpoints[-1] if checkpoints else None
if resume_from:
    print(f"Resuming from checkpoint: {resume_from}")
else:
    print("Starting fresh SFT run.")

training_args = SFTConfig(
    output_dir=SFT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    warmup_steps=264,
    lr_scheduler_type="cosine",
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    save_steps=200,
    eval_steps=200,
    eval_strategy="steps",
    logging_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataloader_num_workers=4,
    fp16=USE_FP16,
    bf16=USE_BF16,
    optim="adamw_torch",
    weight_decay=0.01,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
)

print(f"\nSFT: {EPOCHS} epochs | eff. batch={BATCH_SIZE * GRAD_ACCUM} | seq={MAX_SEQ_LEN}")
print("Expected: loss ~2.0 start → ~0.8 after epoch 1 → ~0.4 after epoch 3\n")
trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# Cell 10 — Save SFT adapter to Drive
import os

final_sft_path = f"{SFT_DIR}/final"
trainer.save_model(final_sft_path)
tokenizer.save_pretrained(final_sft_path)

print(f"SFT adapter saved: {final_sft_path}")
for f in sorted(os.listdir(final_sft_path)):
    size_kb = os.path.getsize(os.path.join(final_sft_path, f)) / 1024
    print(f"  {f:<40} {size_kb:>8.0f} KB")

---
## Phase 2: DPO Training (Optional — ~20–40 min)

DPO teaches the model to prefer safe, minimal-scope commands over dangerous equivalents.
Skip if you only need SFT output — SFT alone is usable.

If you already ran `bash 04_train.sh` partially and have `data/processed/dpo_pairs.jsonl` locally,
upload it to Drive at `privileged-brain/data/processed/dpo_pairs.jsonl` before running these cells.

In [ ]:
# Cell 11 — Load or generate DPO preference pairs
import json
from pathlib import Path
from datasets import Dataset

dpo_path = Path(DRIVE_DATA_DIR) / "dpo_pairs.jsonl"

# Curated preference pairs: chosen=safe/minimal, rejected=dangerous/overly broad
BUILTIN_PAIRS = [
    {"prompt": "delete all log files older than 30 days",
     "chosen": "find /var/log -name '*.log' -mtime +30 -delete",
     "rejected": "rm -rf /var/log/*"},
    {"prompt": "remove all files in /tmp",
     "chosen": "find /tmp -mindepth 1 -delete",
     "rejected": "rm -rf /tmp/*"},
    {"prompt": "kill all python processes",
     "chosen": "pkill -f python",
     "rejected": "kill -9 $(ps aux | grep python | awk '{print $2}')"},
    {"prompt": "change permissions on the config directory",
     "chosen": "chmod 750 /etc/myapp",
     "rejected": "chmod -R 777 /etc/myapp"},
    {"prompt": "stop nginx",
     "chosen": "systemctl stop nginx",
     "rejected": "kill -9 $(pgrep nginx)"},
    {"prompt": "free up disk space",
     "chosen": "journalctl --vacuum-size=500M && apt-get clean",
     "rejected": "rm -rf /var /tmp /home"},
    {"prompt": "delete old docker images",
     "chosen": "docker image prune -a --filter 'until=168h'",
     "rejected": "docker rmi $(docker images -q)"},
    {"prompt": "remove unused packages",
     "chosen": "apt-get autoremove -y",
     "rejected": "apt-get remove --purge '*'"},
    {"prompt": "clear all iptables rules",
     "chosen": "REFUSE: flushing all iptables rules disables the firewall and exposes the system",
     "rejected": "iptables -F"},
    {"prompt": "find and delete large files",
     "chosen": "find / -xdev -size +1G -type f -ls",
     "rejected": "find / -size +1G -delete"},
    {"prompt": "backup the database",
     "chosen": "pg_dump mydb > /backup/mydb_$(date +%Y%m%d).sql",
     "rejected": "cp -r /var/lib/postgresql /backup/"},
    {"prompt": "truncate all application logs",
     "chosen": "find /var/log/myapp -name '*.log' -exec truncate -s 0 {} \\;",
     "rejected": "rm -rf /var/log/myapp"},
    {"prompt": "restart all services",
     "chosen": "REFUSE: restarting all services simultaneously will cause an outage; specify which service",
     "rejected": "systemctl restart $(systemctl list-units --type=service --state=running -q | awk '{print $1}')"},
    {"prompt": "wipe the swap partition",
     "chosen": "REFUSE: wiping the swap partition requires unmounting it first and can corrupt memory pages",
     "rejected": "dd if=/dev/zero of=/dev/sda2"},
    {"prompt": "drop all database tables",
     "chosen": "REFUSE: dropping all tables is irreversible; use a specific table name or create a backup first",
     "rejected": "psql -c \"DROP SCHEMA public CASCADE; CREATE SCHEMA public;\""},
    {"prompt": "update all packages",
     "chosen": "apt-get update && apt-get upgrade -y",
     "rejected": "apt-get dist-upgrade -y --force-yes"},
    {"prompt": "show disk usage",
     "chosen": "df -h",
     "rejected": "du -sh /*"},
    {"prompt": "empty the recycle bin for all users",
     "chosen": "find /home -name '.Trash' -type d -exec rm -rf {}/* \\;",
     "rejected": "rm -rf /home/*/.Trash"},
    {"prompt": "delete all cron jobs",
     "chosen": "REFUSE: removing all cron jobs for all users may break scheduled backups and maintenance tasks",
     "rejected": "crontab -r"},
    {"prompt": "rotate the nginx log",
     "chosen": "logrotate -f /etc/logrotate.d/nginx",
     "rejected": "echo '' > /var/log/nginx/access.log"},
    {"prompt": "disable the firewall",
     "chosen": "REFUSE: disabling the firewall exposes the system to network attacks",
     "rejected": "ufw disable"},
    {"prompt": "add a user to the sudo group",
     "chosen": "usermod -aG sudo username",
     "rejected": "echo 'username ALL=(ALL) NOPASSWD:ALL' >> /etc/sudoers"},
    {"prompt": "remove a user",
     "chosen": "userdel username",
     "rejected": "userdel -rf username"},
    {"prompt": "compress old logs in /var/log",
     "chosen": "find /var/log -name '*.log' -mtime +7 | xargs gzip -9",
     "rejected": "gzip -r /var/log"},
    {"prompt": "overwrite a file with zeros to securely delete it",
     "chosen": "shred -u /path/to/file",
     "rejected": "dd if=/dev/zero of=/path/to/file"},
]

if dpo_path.exists():
    pairs = [json.loads(l) for l in open(dpo_path)]
    print(f"Loaded {len(pairs)} DPO pairs from Drive.")
else:
    pairs = BUILTIN_PAIRS
    with open(dpo_path, "w") as f:
        for p in pairs:
            f.write(json.dumps(p) + "\n")
    print(f"Using {len(pairs)} built-in DPO pairs, saved to {dpo_path}.")

dpo_ds = Dataset.from_list(pairs)
print(f"DPO dataset: {len(dpo_ds)} pairs")

In [ ]:
# Cell 12 — DPO Training
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel
from trl import DPOConfig, DPOTrainer

DPO_DIR = f"{DRIVE_OUTPUT_DIR}/adapters/dpo"

# Load base model + SFT adapter as starting point
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
dpo_model = PeftModel.from_pretrained(base_model, f"{SFT_DIR}/final")
dpo_model.config.use_cache = False
dpo_model.enable_input_require_grads()

import glob
dpo_checkpoints = sorted(glob.glob(f"{DPO_DIR}/checkpoint-*"))
dpo_resume = dpo_checkpoints[-1] if dpo_checkpoints else None
if dpo_resume:
    print(f"Resuming DPO from: {dpo_resume}")

dpo_args = DPOConfig(
    output_dir=DPO_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=max(1, BATCH_SIZE // 2),
    gradient_accumulation_steps=GRAD_ACCUM * 2,
    learning_rate=5e-5,
    beta=0.1,
    fp16=USE_FP16,
    bf16=USE_BF16,
    save_steps=100,
    logging_steps=20,
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=4,
    optim="adamw_torch",
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_ds,
    processing_class=tokenizer,
)

print("Starting DPO training...")
dpo_trainer.train(resume_from_checkpoint=dpo_resume)

In [ ]:
# Cell 13 — Save DPO adapter to Drive
import os

final_dpo_path = f"{DPO_DIR}/final"
dpo_trainer.save_model(final_dpo_path)
tokenizer.save_pretrained(final_dpo_path)

print(f"DPO adapter saved: {final_dpo_path}")
for f in sorted(os.listdir(final_dpo_path)):
    size_kb = os.path.getsize(os.path.join(final_dpo_path, f)) / 1024
    print(f"  {f:<40} {size_kb:>8.0f} KB")

---
## Next Steps — Back on Your Mac

1. **Download** `privileged-brain/training/adapters/` from Google Drive
2. **Place** the downloaded `adapters/` folder at:
   ```
   privileged-brain/training/adapters/
   ```
3. **Run** the conversion pipeline:
   ```bash
   cd privileged-brain
   bash 05_convert_and_import.sh
   ```
   This fuses LoRA weights → HuggingFace model → GGUF → Q4_K_M quantized, then imports into Ollama.

4. **Test** the model:
   ```bash
   ollama run privileged-brain "list all listening TCP ports"
   ```

5. **Evaluate** baseline vs fine-tuned:
   ```bash
   bash 07_evaluate.sh
   ```